# MBA03 · Week 7 — The founder does not know the numbers
**European School of Economics · Mon 9 Nov 2026 · Tutor: Vincenzo Nardelli**

In weeks 1–5 every number at **Arno Bikes** was given: a fixed cost, a list price, a coupon. This quarter the founder asks four questions she cannot answer with a given number — how many customers do both, which supplier sent the faulty battery, how many returns next week, what range to print on the box. Each answer comes with a spread.

This notebook is driven by the tutor. Every picture is followed by the same computation **by hand on paper**: formula, substitution, number, one sentence for the Board. The code is optional and not examined.

In [1]:
# Setup (Colab has plotly preinstalled; this is a no-op there)
import subprocess, sys
try:
    import plotly
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotly"])
import math
import numpy as np, pandas as pd
from scipy import stats
import plotly.graph_objects as go

RED, GOLD, GREY = "#AF1F25", "#CDBA80", "#7a7f85"
print("ready")

ready


## A. How many of our customers do both? — sets and counting

The CRM says: of **200** customers, **120** bought accessories, **90** a service plan, **40** neither. The founder wants to upsell the ones who bought **both**.

$$n(A \cup S) = n(A) + n(S) - n(A \cap S)$$

*Business reading:* "or" adds the two counts, then takes the overlap out once — otherwise the "both" customers are counted twice.

Then counting: eight people work at Arno Bikes, three go to the Milan trade fair. **Does swapping two people change the outcome?** No → combinations. Yes (named roles) → permutations.

In [2]:
def venn_count(N, nA, nB, neither, A="accessories", B="service plan"):
    # Four regions of a two-circle Venn diagram, from the four facts we usually have
    at_least_one = N - neither
    both = nA + nB - at_least_one          # the addition law, solved for the overlap
    only_A, only_B = nA - both, nB - both
    rows = [(f"{A} only", only_A), ("both", both), (f"{B} only", only_B), ("neither", neither)]
    print(f"at least one = {N} − {neither} = {at_least_one}")
    print(f"both = {nA} + {nB} − {at_least_one} = {both}\n")
    print(f"{'region':<20} {'count':>6} {'share':>7}")
    for name, n in rows:
        print(f"{name:<20} {n:>6} {n/N:>7.2f}")
    print(f"{'total':<20} {sum(n for _, n in rows):>6} {sum(n for _, n in rows)/N:>7.2f}")
    assert sum(n for _, n in rows) == N
    return both

both = venn_count(200, 120, 90, 40)
P_union = 120/200 + 90/200 - both/200
print(f"\nP(A or S) = .60 + .45 − {both/200:.2f} = {P_union:.2f};  complement = {1 - P_union:.2f}")

at least one = 200 − 40 = 160
both = 120 + 90 − 160 = 50

region                count   share
accessories only         70    0.35
both                     50    0.25
service plan only        40    0.20
neither                  40    0.20
total                   200    1.00

P(A or S) = .60 + .45 − 0.25 = 0.80;  complement = 0.20


In [3]:
n, k = 8, 3
teams = math.comb(n, k)                   # order does not matter
roles = math.perm(n, k)                   # stand lead, demo rider, sales
print(pd.DataFrame({
    "question": ["teams of 3 (combinations)", "3 named roles (permutations)", "ratio"],
    "formula": ["8! / (3!·5!) = 8·7·6 / 6", "8! / 5! = 8·7·6", "3! = 6"],
    "result": [teams, roles, roles // teams]}).to_string(index=False))
assert roles == teams * math.factorial(k)
print(f"\nEach team of {k} can be arranged in {k}! = {math.factorial(k)} ways: {teams} × {math.factorial(k)} = {roles}.")

                    question                  formula  result
   teams of 3 (combinations) 8! / (3!·5!) = 8·7·6 / 6      56
3 named roles (permutations)          8! / 5! = 8·7·6     336
                       ratio                   3! = 6       6

Each team of 3 can be arranged in 3! = 6 ways: 56 × 6 = 336.


In [4]:
ks = list(range(0, 9))
fig = go.Figure(go.Bar(x=ks, y=[math.comb(8, j) for j in ks], marker_color=[RED if j == 3 else GREY for j in ks],
                       text=[math.comb(8, j) for j in ks], textposition="outside"))
fig.update_layout(template="simple_white", height=420, title="How many groups of k can you pick from 8 people?",
                  xaxis_title="k, size of the group", yaxis_title="C(8, k)", yaxis_range=[0, 80])
fig.show()
print("Symmetric: choosing the 3 who go is the same as choosing the 5 who stay. C(8,3) = C(8,5) =", math.comb(8, 5))

Symmetric: choosing the 3 who go is the same as choosing the 5 who stay. C(8,3) = C(8,5) = 56


### 🔍 CHECK
The four regions must add back to **200**, and the shares to **1.00**. On paper, if they do not, stop before you compute any probability. For counting, ask the one question first: *does order matter?*

## B. Which supplier sent the faulty battery? — the two-way table and Bayes

Last month: **1,000** batteries. **Volta** sent **700** at a **2%** defect rate; **Ampere** sent **300** at **5%**. A faulty battery is on the bench, label gone.

$$P(A \mid B) = \frac{\text{number in both } A \text{ and } B}{\text{number in } B}$$

*Business reading:* "given B" means look only at B's row (or column) and divide by **its** total. Same cell on top, different total underneath, different question.

Then the independence check: supplier and fault are **independent** if every row shows the same share of faulty as the total. Under independence the cells would be *row total × column share*.

In [5]:
def two_way(suppliers, counts, rates, bad="faulty", good="good"):
    # Build the table of counts from the three facts: who sent how many, at what defect rate
    faulty = [round(c * r) for c, r in zip(counts, rates)]
    tab = pd.DataFrame({good: [c - f for c, f in zip(counts, faulty)], bad: faulty}, index=suppliers)
    tab["total"] = tab[good] + tab[bad]
    tab.loc["total"] = tab.sum()
    return tab

for s, c, r in zip(["Volta", "Ampere"], [700, 300], [.02, .05]):
    print(f"{s}: {c} × {r:.0%} = {round(c*r)} faulty, {c} − {round(c*r)} = {c - round(c*r)} good")
tab = two_way(["Volta", "Ampere"], [700, 300], [.02, .05])
print("\n", tab.to_string(), sep="")

print("\nThe ROW — given the supplier, how often faulty?")
for s in ["Volta", "Ampere"]:
    print(f"  P(faulty | {s:<6}) = {tab.loc[s, 'faulty']:>2} / {tab.loc[s, 'total']:>4} = {tab.loc[s, 'faulty']/tab.loc[s, 'total']:.3f}")
print("  (the defect rates we started from: the table is built right)")
print("\nThe COLUMN — given faulty, which supplier?")
for s in ["Volta", "Ampere"]:
    print(f"  P({s:<6} | faulty) = {tab.loc[s, 'faulty']:>2} / {tab.loc['total', 'faulty']:>4} = {tab.loc[s, 'faulty']/tab.loc['total', 'faulty']:.2f}")
print(f"\nOverall P(faulty) = 29 / 1,000 = {tab.loc['total', 'faulty']/tab.loc['total', 'total']:.3f}")

Volta: 700 × 2% = 14 faulty, 700 − 14 = 686 good
Ampere: 300 × 5% = 15 faulty, 300 − 15 = 285 good

        good  faulty  total
Volta    686      14    700
Ampere   285      15    300
total    971      29   1000

The ROW — given the supplier, how often faulty?
  P(faulty | Volta ) = 14 /  700 = 0.020
  P(faulty | Ampere) = 15 /  300 = 0.050
  (the defect rates we started from: the table is built right)

The COLUMN — given faulty, which supplier?
  P(Volta  | faulty) = 14 /   29 = 0.48
  P(Ampere | faulty) = 15 /   29 = 0.52

Overall P(faulty) = 29 / 1,000 = 0.029


In [6]:
p_faulty = tab.loc["total", "faulty"] / tab.loc["total", "total"]
expected = pd.DataFrame({
    "good":   [tab.loc[s, "total"] * (1 - p_faulty) for s in ["Volta", "Ampere"]],
    "faulty": [tab.loc[s, "total"] * p_faulty for s in ["Volta", "Ampere"]]}, index=["Volta", "Ampere"])
print("Observed share faulty by row:", {s: round(tab.loc[s, "faulty"] / tab.loc[s, "total"], 3) for s in ["Volta", "Ampere"]},
      f" overall {p_faulty:.3f}")
print("\nIf supplier told us nothing about the fault, the table would read:")
print(expected.round(1).to_string())
print("\nObserved faulty: Volta 14 vs 20.3 expected, Ampere 15 vs 8.7 expected.")
print("The rows differ from the total: supplier and fault are DEPENDENT. Say the comparison, then the word.")

Observed share faulty by row: {'Volta': 0.02, 'Ampere': 0.05}  overall 0.029

If supplier told us nothing about the fault, the table would read:
         good  faulty
Volta   679.7    20.3
Ampere  291.3     8.7

Observed faulty: Volta 14 vs 20.3 expected, Ampere 15 vs 8.7 expected.
The rows differ from the total: supplier and fault are DEPENDENT. Say the comparison, then the word.


### Bayes: the faulty column, with shares instead of counts
The founder's prior: "probably Volta, they send 70%". One faulty battery is the evidence.

$$P(\text{Ampere} \mid F) = \frac{P(F \mid \text{Ampere})\,P(\text{Ampere})}{P(F)}$$

Then **drag the slider:** Ampere's defect rate, from 1% to 10%. Volta stays at 2% with 70% of deliveries. When does Ampere become the clear suspect?

In [7]:
def bayes_table(suppliers, prior, rate):
    joint = [p * r for p, r in zip(prior, rate)]           # share of ALL batteries that are faulty from each
    total = sum(joint)
    t = pd.DataFrame({"prior share": prior, "× defect rate": rate, "= faulty, share of all": joint,
                      "÷ total faulty": [j / total for j in joint]}, index=suppliers)
    t.loc["total"] = [sum(prior), np.nan, total, 1.0]
    return t

bt = bayes_table(["Volta", "Ampere"], [.70, .30], [.02, .05])
print(bt.round(3).to_string())
print(f"\nP(Ampere | F) = .05 × .30 / (.02 × .70 + .05 × .30) = .015 / .029 = {bt.loc['Ampere', '÷ total faulty']:.2f}")
print("Before the evidence Ampere was .30; after, .52. A 30% supplier at twice the defect rate is only a coin-flip suspect.")

        prior share  × defect rate  = faulty, share of all  ÷ total faulty
Volta           0.7           0.02                   0.014           0.483
Ampere          0.3           0.05                   0.015           0.517
total           1.0            NaN                   0.029           1.000

P(Ampere | F) = .05 × .30 / (.02 × .70 + .05 × .30) = .015 / .029 = 0.52
Before the evidence Ampere was .30; after, .52. A 30% supplier at twice the defect rate is only a coin-flip suspect.


In [8]:
rates = np.round(np.arange(0.01, 0.1001, 0.005), 3)
def posterior(rA, rV=.02, pV=.70):
    jV, jA = pV * rV, (1 - pV) * rA
    return jV / (jV + jA), jA / (jV + jA)

fig = go.Figure()
for r in rates:
    pV, pA = posterior(r)
    fig.add_bar(x=["Volta", "Ampere"], y=[pV, pA], marker_color=[GREY, RED], visible=bool(abs(r - .05) < 1e-9),
                text=[f"{pV:.2f}", f"{pA:.2f}"], textposition="outside", name=f"{r:.1%}")
fig.add_hline(y=.30, line=dict(color=GOLD, dash="dot"), annotation_text="Ampere's prior share .30")
steps = [dict(method="update", label=f"{r:.1%}",
              args=[{"visible": [j == i for j in range(len(rates))]},
                    {"title": f"Ampere defect rate {r:.1%} → P(Ampere | faulty) = {posterior(r)[1]:.2f}"}])
         for i, r in enumerate(rates)]
fig.update_layout(template="simple_white", height=460, showlegend=False, yaxis_range=[0, 1],
                  title="Ampere defect rate 5.0% → P(Ampere | faulty) = 0.52", yaxis_title="P(supplier | faulty)",
                  sliders=[dict(active=int(np.argmin(abs(rates - .05))), steps=steps,
                                currentvalue=dict(prefix="Ampere's defect rate: "))])
fig.show()
print(f"{'Ampere rate':>12} {'P(Ampere | faulty)':>20}")
for r in (.01, .02, .05, .08, .10):
    print(f"{r:>12.0%} {posterior(r)[1]:>20.2f}")

 Ampere rate   P(Ampere | faulty)
          1%                 0.18
          2%                 0.30
          5%                 0.52
          8%                 0.63
         10%                 0.68


### 🔍 CHECK
At 2% both rows read the same and the posterior equals the prior (.30): the supplier tells you nothing. The row check — $14/700 = .02$ — must return the defect rate you were given. If it does not, the table is wrong, not the question.

## C. How many returns next week? — mean and variance, binomial, Poisson

Five weeks of warranty returns: **4, 7, 3, 6, 5**.

$$\bar{x} = \frac{\sum x}{n} \qquad s^{2} = \frac{\sum (x - \bar{x})^{2}}{n-1}$$

*Business reading:* the mean is the total, shared equally. The variance is the average squared distance from it — divided by $n-1$ because the deviations always sum to zero, so only $n-1$ of them are free.

Then the **random variable**: next week's returns, with a share attached to each value — twenty weeks of history turned into shares. Same two formulas, with the shares as weights instead of $1/n$: $E(x) = \sum x\,P(x)$ and $\sigma^{2} = \sum (x - \mu)^{2}\,P(x)$.

In [9]:
def spread_table(x, label="returns"):
    x = np.array(x, dtype=float); n = len(x); xbar = x.sum() / n
    t = pd.DataFrame({"week": range(1, n + 1), label: x, "x − x̄": x - xbar, "(x − x̄)²": (x - xbar) ** 2})
    t.loc["sum"] = ["", x.sum(), (x - xbar).sum(), ((x - xbar) ** 2).sum()]
    print(t.to_string(index=False))
    ss = ((x - xbar) ** 2).sum()
    print(f"\nmean   x̄  = {x.sum():.0f} / {n} = {xbar:g}")
    print(f"variance s² = {ss:.0f} / ({n} − 1) = {ss:.0f} / {n-1} = {ss/(n-1):g}")
    print(f"st. dev. s  = √{ss/(n-1):g} = {math.sqrt(ss/(n-1)):.2f}")
    return xbar, ss / (n - 1)

xbar, s2 = spread_table([4, 7, 3, 6, 5])
print(f"\nReading: a typical week is {xbar:g} returns, give or take about {math.sqrt(s2):.1f}. The deviations sum to zero: a check, not a result.")

week  returns  x − x̄  (x − x̄)²
   1      4.0    -1.0        1.0
   2      7.0     2.0        4.0
   3      3.0    -2.0        4.0
   4      6.0     1.0        1.0
   5      5.0     0.0        0.0
         25.0     0.0       10.0

mean   x̄  = 25 / 5 = 5
variance s² = 10 / (5 − 1) = 10 / 4 = 2.5
st. dev. s  = √2.5 = 1.58

Reading: a typical week is 5 returns, give or take about 1.6. The deviations sum to zero: a check, not a result.


In [10]:
def rv_table(values, shares):
    v, P = np.array(values, dtype=float), np.array(shares, dtype=float)
    assert abs(P.sum() - 1) < 1e-9, "shares must add to 1"
    mu = (v * P).sum()
    t = pd.DataFrame({"x": v, "P(x)": P, "x·P(x)": v * P, "x − μ": v - mu, "(x − μ)²": (v - mu) ** 2,
                      "(x − μ)²·P(x)": (v - mu) ** 2 * P})
    t.loc["sum"] = ["", P.sum(), mu, "", "", ((v - mu) ** 2 * P).sum()]
    print(t.to_string(index=False))
    var = ((v - mu) ** 2 * P).sum()
    print(f"\nE(x) = {' + '.join(f'{a:g}({b:.2f})' for a, b in zip(v, P))} = {mu:.1f}")
    print(f"σ² = {var:.2f},  σ = √{var:.2f} = {math.sqrt(var):.2f}")
    return mu, var

mu_rv, var_rv = rv_table([3, 4, 5, 6, 7], [.15, .20, .30, .20, .15])
print(f"\nReading: expect {mu_rv:.0f} returns a week, give or take about {math.sqrt(var_rv):.1f}. No single week has exactly 5.0; over 50 weeks, about 250.")

   x  P(x)  x·P(x) x − μ (x − μ)²  (x − μ)²·P(x)
 3.0  0.15    0.45  -2.0      4.0            0.6
 4.0  0.20    0.80  -1.0      1.0            0.2
 5.0  0.30    1.50   0.0      0.0            0.0
 6.0  0.20    1.20   1.0      1.0            0.2
 7.0  0.15    1.05   2.0      4.0            0.6
      1.00    5.00                           1.6

E(x) = 3(0.15) + 4(0.20) + 5(0.30) + 6(0.20) + 7(0.15) = 5.0
σ² = 1.60,  σ = √1.60 = 1.26

Reading: expect 5 returns a week, give or take about 1.3. No single week has exactly 5.0; over 50 weeks, about 250.


### Binomial: twelve test rides, how many buy?
Saturday: **12** test rides, one rider in four buys ($p = .25$), each decides alone. A fixed number of yes/no trials, same $p$, independent.

$$P(x) = \binom{n}{x}\, p^{x}\, (1-p)^{n-x}$$

Below the four steps, **two sliders, one at a time:** the number of test rides $n$ (with $p = .25$) and the share who buy $p$ (with $n = 12$). Push $n$ to 40 and watch the bars become a bell; drop $p$ to .05 and see where the peak goes.

In [11]:
def binomial_steps(n, x, p):
    ways = math.comb(n, x); yes = p ** x; no = (1 - p) ** (n - x)
    print(f"1. the ways to pick which {x} of the {n} buy:   C({n},{x}) = {ways}")
    print(f"2. those {x} buy:                              {p}^{x} = {yes:.6f}")
    print(f"3. the other {n-x} do not:                      {1-p}^{n-x} = {no:.4f}")
    print(f"4. multiply:                                 {ways} × {yes:.6f} × {no:.4f} = {ways*yes*no:.3f}")
    return ways * yes * no

n, p = 12, .25
P3 = binomial_steps(n, 3, p)
print(f"\nscipy agrees: binom.pmf(3, 12, .25) = {stats.binom.pmf(3, n, p):.3f}")
print(f"\nmean          np = {n} × {p} = {n*p:g} buyers")
print(f"st. dev.  √np(1−p) = √({n} × {p} × {1-p}) = √{n*p*(1-p):g} = {math.sqrt(n*p*(1-p)):g}")
P0 = (1 - p) ** n
print(f"nobody buys   P(0) = {1-p}^{n} = {P0:.3f}")
print(f"at least one  1 − P(0) = {1 - P0:.3f}")
print("\nReading: one Saturday in four ends with exactly three buyers; nobody buying is a one-in-thirty Saturday.")

1. the ways to pick which 3 of the 12 buy:   C(12,3) = 220
2. those 3 buy:                              0.25^3 = 0.015625
3. the other 9 do not:                      0.75^9 = 0.0751
4. multiply:                                 220 × 0.015625 × 0.0751 = 0.258

scipy agrees: binom.pmf(3, 12, .25) = 0.258

mean          np = 12 × 0.25 = 3 buyers
st. dev.  √np(1−p) = √(12 × 0.25 × 0.75) = √2.25 = 1.5
nobody buys   P(0) = 0.75^12 = 0.032
at least one  1 − P(0) = 0.968

Reading: one Saturday in four ends with exactly three buyers; nobody buying is a one-in-thirty Saturday.


In [12]:
ns = list(range(4, 41, 2)); ps = np.round(np.arange(0.05, 0.96, 0.05), 2)

def pmf_bars(n, p, visible):
    xs = np.arange(0, n + 1); ys = stats.binom.pmf(xs, n, p)
    return [go.Bar(x=xs, y=ys, marker_color=RED, visible=visible, name=f"n={n}, p={p}", showlegend=False),
            go.Scatter(x=[n * p, n * p], y=[0, ys.max() * 1.08], mode="lines", visible=visible, showlegend=False,
                       line=dict(color=GOLD, width=3, dash="dash"), name=f"mean {n*p:.1f}")]

fig = go.Figure()
for n_ in ns:
    fig.add_traces(pmf_bars(n_, .25, n_ == 12))
for p_ in ps:
    fig.add_traces(pmf_bars(12, p_, False))
T = 2 * (len(ns) + len(ps))
def vis(idx):                       # show the pair of traces number idx, hide all others
    v = [False] * T; v[2*idx] = v[2*idx + 1] = True; return v
steps_n = [dict(method="update", label=str(n_), args=[{"visible": vis(i)},
           {"title": f"n = {n_} test rides, p = .25 → mean np = {n_*.25:.1f}, σ = {math.sqrt(n_*.25*.75):.2f}"}]) for i, n_ in enumerate(ns)]
steps_p = [dict(method="update", label=f"{p_:.2f}", args=[{"visible": vis(len(ns) + i)},
           {"title": f"n = 12 test rides, p = {p_:.2f} → mean np = {12*p_:.1f}, σ = {math.sqrt(12*p_*(1-p_)):.2f}"}]) for i, p_ in enumerate(ps)]
fig.update_layout(template="simple_white", height=520, title="n = 12 test rides, p = .25 → mean np = 3.0, σ = 1.50",
                  xaxis_title="number of buyers", yaxis_title="probability", xaxis_range=[-0.5, 40.5],
                  sliders=[dict(active=ns.index(12), steps=steps_n, currentvalue=dict(prefix="n (p = .25): "), y=-0.05),
                           dict(active=int(np.argmin(abs(ps - .25))), steps=steps_p, currentvalue=dict(prefix="p (n = 12): "), y=-0.35)],
                  margin=dict(b=200))
fig.show()

### Poisson: the helpline, three calls an hour
Calls arrive at random, **3 an hour**. The one person answering steps out for **half an hour**. A count in an interval, constant rate, no ceiling. Write $\mu$ = rate × interval **before** any number.

$$P(x) = \frac{\mu^{x} e^{-\mu}}{x!}$$

In [13]:
def poisson_steps(rate, hours):
    mu = rate * hours
    print(f"μ = {rate} an hour × {hours:g} hour(s) = {mu:g}")
    P0 = math.exp(-mu)
    print(f"  P(0)  = {mu:g}^0 e^−{mu:g} / 0! = e^−{mu:g} = {P0:.3f}")
    print(f"  P(≥1) = 1 − {P0:.3f} = {1 - P0:.3f}")
    print(f"  P(2)  = {mu:g}² e^−{mu:g} / 2! = {mu**2:g} × {P0:.4f} / 2 = {stats.poisson.pmf(2, mu):.3f}")
    return mu, P0

print("A full hour:"); mu_h, P0_h = poisson_steps(3, 1)
print("\nThe half hour she is away — rescale first:"); mu_half, P0_half = poisson_steps(3, 0.5)
print(f"\nWithout the rescaling you would say {P0_h:.0%} for the half hour. It is {P0_half:.0%}: four times out. μ first, always.")
print(f"Reading: leaving the helpline for half an hour, we miss at least one call {1 - P0_half:.0%} of the time — three times out of four.")

A full hour:
μ = 3 an hour × 1 hour(s) = 3
  P(0)  = 3^0 e^−3 / 0! = e^−3 = 0.050
  P(≥1) = 1 − 0.050 = 0.950
  P(2)  = 3² e^−3 / 2! = 9 × 0.0498 / 2 = 0.224

The half hour she is away — rescale first:
μ = 3 an hour × 0.5 hour(s) = 1.5
  P(0)  = 1.5^0 e^−1.5 / 0! = e^−1.5 = 0.223
  P(≥1) = 1 − 0.223 = 0.777
  P(2)  = 1.5² e^−1.5 / 2! = 2.25 × 0.2231 / 2 = 0.251

Without the rescaling you would say 5% for the half hour. It is 22%: four times out. μ first, always.
Reading: leaving the helpline for half an hour, we miss at least one call 78% of the time — three times out of four.


In [14]:
mus = np.round(np.arange(0.5, 10.01, 0.5), 1)
xs = np.arange(0, 21)
fig = go.Figure()
for m in mus:
    fig.add_bar(x=xs, y=stats.poisson.pmf(xs, m), marker_color=RED, visible=bool(m == 3.0), name=f"μ={m}", showlegend=False)
steps = [dict(method="update", label=f"{m:g}", args=[{"visible": [j == i for j in range(len(mus))]},
         {"title": f"μ = {m:g} expected calls → P(0) = {math.exp(-m):.3f}, P(≥1) = {1 - math.exp(-m):.3f}"}]) for i, m in enumerate(mus)]
fig.update_layout(template="simple_white", height=460, title="μ = 3 expected calls → P(0) = 0.050, P(≥1) = 0.950",
                  xaxis_title="calls in the interval", yaxis_title="probability",
                  sliders=[dict(active=int(np.where(mus == 3.0)[0][0]), steps=steps, currentvalue=dict(prefix="μ = rate × interval: "))])
fig.show()
print("Mean and variance of a Poisson are both μ: a high rate gives a wide, bell-shaped pile. Push μ to 10 and look.")

Mean and variance of a Poisson are both μ: a high rate gives a wide, bell-shaped pile. Push μ to 10 and look.


### 🔍 CHECK
Deviations sum to zero; shares sum to one; $P(0) + P(\geq 1) = 1$. Before any formula, say which model in words: *how many of these $n$?* is binomial; *how many in this interval?* is Poisson, and $\mu$ comes first.

## D. What range do we promise? — the normal

Engineering measured 200 bikes on the same loop: **90 km on average, standard deviation 8 km**, bell-shaped. $X \sim N(90, 8)$. Marketing wants to print **75 km**.

$$z = \frac{x - \mu}{\sigma}$$

Four steps, every time: **sketch and shade → standardise → table → sentence.** The table gives the area to the **left** of $z$.

Below the steps, **three sliders, one at a time** from the baseline $N(90, 8)$, threshold 75: the mean $\mu$, the spread $\sigma$, and the threshold printed on the box. The shaded tail is the share of riders who see less than promised. Then the empirical rule — one σ, two σ, three σ: the same three numbers for every bell, which is why we measure distance in σ and not in km.

In [15]:
def normal_steps(x, mu, sigma, tail="below"):
    z = (x - mu) / sigma
    z2 = round(z, 2)                                       # the table has two decimals
    left = round(stats.norm.cdf(z2), 4)                    # what the printed table shows
    print(f"1. sketch: the bell centred at {mu}, shade the area {tail} {x}")
    print(f"2. standardise: z = ({x} − {mu}) / {sigma} = {x - mu} / {sigma} = {z:.3f} ≈ {z2:.2f}")
    print(f"3. table, area to the left of {z2:.2f}: {left:.4f}   (Excel: =NORM.S.DIST({z2:.2f}, TRUE))")
    area = left if tail == "below" else 1 - left
    if tail == "above":
        print(f"   the whole area is 1, so P(X > {x}) = 1 − {left:.4f} = {area:.3f}")
    print(f"4. P(X {'<' if tail == 'below' else '>'} {x}) = {area:.3f}")
    return z, area

print("Print 75 km: what share of riders see less?"); z75, below75 = normal_steps(75, 90, 8)
print(f"   Sentence: print 75 km and about {round(below75*100):.0f} riders in a hundred will see less than the box promised.\n")
print("And the share above 75 km:"); _, above75 = normal_steps(75, 90, 8, tail="above")

print("\nBackwards — the Board accepts 5% below the box. What do we print?")
z05 = -1.645                                                # .05 sits between z = −1.64 (.0505) and −1.65 (.0495)
x05 = 90 + z05 * 8
print(f"   table, backwards: area .05 → z = {z05}   (Excel: =NORM.S.INV(0.05) = {stats.norm.ppf(.05):.3f})")
print(f"   undo the standardisation: x = μ + zσ = 90 + ({z05}) × 8 = 90 − {abs(z05)*8:.2f} = {x05:.2f} km")
print(f"   Print 76 km, a promise rounds down: the share below it is P(z < {(76-90)/8:.2f}) = {stats.norm.cdf(76, 90, 8):.3f}.")
print("   Sentence: print 76 km; about one rider in twenty-five will see less, and every km added to the box adds one or two complaints in a hundred.")

Print 75 km: what share of riders see less?
1. sketch: the bell centred at 90, shade the area below 75
2. standardise: z = (75 − 90) / 8 = -15 / 8 = -1.875 ≈ -1.88
3. table, area to the left of -1.88: 0.0301   (Excel: =NORM.S.DIST(-1.88, TRUE))
4. P(X < 75) = 0.030
   Sentence: print 75 km and about 3 riders in a hundred will see less than the box promised.

And the share above 75 km:
1. sketch: the bell centred at 90, shade the area above 75
2. standardise: z = (75 − 90) / 8 = -15 / 8 = -1.875 ≈ -1.88
3. table, area to the left of -1.88: 0.0301   (Excel: =NORM.S.DIST(-1.88, TRUE))
   the whole area is 1, so P(X > 75) = 1 − 0.0301 = 0.970
4. P(X > 75) = 0.970

Backwards — the Board accepts 5% below the box. What do we print?
   table, backwards: area .05 → z = -1.645   (Excel: =NORM.S.INV(0.05) = -1.645)
   undo the standardisation: x = μ + zσ = 90 + (-1.645) × 8 = 90 − 13.16 = 76.84 km
   Print 76 km, a promise rounds down: the share below it is P(z < -1.75) = 0.040.
   Sentence: prin

In [16]:
mus_ = list(range(80, 101, 2)); sigmas_ = list(range(4, 17, 2)); thr_ = list(range(70, 91, 1))
grid = np.linspace(50, 130, 400)

def bell(mu, sigma, thr, visible):
    pdf = stats.norm.pdf(grid, mu, sigma); tail = grid <= thr
    return [go.Scatter(x=grid, y=pdf, mode="lines", line=dict(color="#363636", width=3), visible=visible, showlegend=False),
            go.Scatter(x=grid[tail], y=pdf[tail], fill="tozeroy", mode="lines", line=dict(color=RED, width=0),
                       fillcolor="rgba(175,31,37,.45)", visible=visible, showlegend=False)]
settings = [(m, 8, 75) for m in mus_] + [(90, s, 75) for s in sigmas_] + [(90, 8, t) for t in thr_]
fig = go.Figure()
for (m, s, t) in settings:
    fig.add_traces(bell(m, s, t, (m, s, t) == (90, 8, 75)))
T = 2 * len(settings)
def vis(idx):
    v = [False] * T; v[2*idx] = v[2*idx + 1] = True; return v
def title(m, s, t):
    return f"N({m}, {s}), box says {t} km → z = {(t-m)/s:.2f}, share below = {stats.norm.cdf(t, m, s):.3f}"
mk = lambda i, lab: dict(method="update", label=lab, args=[{"visible": vis(i)}, {"title": title(*settings[i])}])
off = 0; steps_mu = [mk(off + i, str(m)) for i, m in enumerate(mus_)]
off += len(mus_); steps_sg = [mk(off + i, str(s)) for i, s in enumerate(sigmas_)]
off += len(sigmas_); steps_th = [mk(off + i, str(t)) for i, t in enumerate(thr_)]
fig.update_layout(template="simple_white", height=600, title=title(90, 8, 75), xaxis_title="range, km", yaxis_title="density",
                  yaxis_range=[0, 0.105], margin=dict(b=260),
                  sliders=[dict(active=mus_.index(90), steps=steps_mu, currentvalue=dict(prefix="μ (σ = 8, box 75): "), y=-0.04),
                           dict(active=sigmas_.index(8), steps=steps_sg, currentvalue=dict(prefix="σ (μ = 90, box 75): "), y=-0.30),
                           dict(active=thr_.index(75), steps=steps_th, currentvalue=dict(prefix="box says, km (μ = 90, σ = 8): "), y=-0.56)])
fig.show()
print(f"{'print':>6} {'share who see less':>20}")
for t in (85, 80, 76.8, 75):
    print(f"{t:>6} {stats.norm.cdf(t, 90, 8):>20.0%}")

 print   share who see less
    85                  27%
    80                  11%
  76.8                   5%
    75                   3%


In [17]:
mu, sigma = 90, 8
print(f"{'within':>8} {'km range':>12} {'share, exact':>13} {'rule of thumb':>14}")
for k, rule in zip((1, 2, 3), ("68%", "95%", "99.7%")):
    share = stats.norm.cdf(k) - stats.norm.cdf(-k)
    print(f"{k:>7}σ {f'{mu - k*sigma} to {mu + k*sigma}':>12} {share:>13.3f} {rule:>14}")
print(f"\nA bike at 74 km is two σ below the mean: {stats.norm.cdf(-2):.3f}, about one in forty.")

  within     km range  share, exact  rule of thumb
      1σ     82 to 98         0.683            68%
      2σ    74 to 106         0.954            95%
      3σ    66 to 114         0.997          99.7%

A bike at 74 km is two σ below the mean: 0.023, about one in forty.


### Skewness: when the bell is the wrong picture
Battery range is symmetric: mean and median coincide. Delivery time is not — most parcels take two days, a few take ten. A long right tail pulls the mean above the median. Before you use the table, ask whether the data are a bell.

In [18]:
rng = np.random.default_rng(7)
ranges = rng.normal(90, 8, 5000)                                  # symmetric: battery range
delivery = rng.lognormal(mean=np.log(2.4), sigma=0.716, size=5000) # right-skewed: delivery days, median 2.4

fig = go.Figure()
fig.add_histogram(x=ranges, nbinsx=60, name="battery range, km", marker_color=GREY, opacity=.8)
fig.add_histogram(x=delivery * 10 + 50, nbinsx=60, name="delivery time (days, rescaled to share the axis)", marker_color=RED, opacity=.7)
fig.update_layout(template="simple_white", height=420, barmode="overlay", title="A bell and a long right tail",
                  xaxis_title="", yaxis_title="count of 5,000", legend=dict(x=.55, y=.95))
fig.show()

for name, x, unit in (("battery range", ranges, "km"), ("delivery time", delivery, "days")):
    skew = stats.skew(x)
    print(f"{name:<14} mean {np.mean(x):6.2f} {unit:<4} median {np.median(x):6.2f} {unit:<4} skewness {skew:+.2f}"
          f"  → {'mean ≈ median: symmetric, the table applies' if abs(skew) < .2 else 'mean > median: positive skew, the table does not apply'}")

battery range  mean  89.85 km   median  89.76 km   skewness +0.03  → mean ≈ median: symmetric, the table applies
delivery time  mean   3.08 days median   2.42 days skewness +3.17  → mean > median: positive skew, the table does not apply


### 🔍 CHECK
The table only gives "to the left". Everything else is $1 -$ or a difference of two lookups; the sketch tells you which. Forwards: $x \to z \to$ area. Backwards: area $\to z \to x$, and the answer ends in km, not in $z$.

## Drills, solved in code

This section is an optional companion: the code is not examined, but reading it is recommended. Every drill in `drills.md` is solved here in Python. Do the drill on paper first, then run the cell and compare your numbers with the output.

In [19]:
# `close` checks our result against the number printed in drills.md
def close(value, expected, tol=1e-3):
    assert abs(value - expected) <= tol, f"got {value}, drills.md says {expected}"

def table_left(z):
    # what the printed standard normal table shows: z to two decimals, area to the left to four
    return round(stats.norm.cdf(round(z, 2)), 4)

print("helpers ready")

helpers ready


**D1.** The founder sends 2 of the 6 mechanics to a training course. (a) How many different pairs? (b) With a **lead** and an **assistant**, how many assignments? (c) Why is (b) exactly twice (a)?

In [20]:
n, k = 6, 2
pairs = math.comb(n, k)                       # (a) a pair is a pair: order does not matter
assignments = math.perm(n, k)                 # (b) named roles: order matters
print(f"(a) C(6,2) = 6! / (2!·4!) = 6·5 / 2 = {6*5} / 2 = {pairs} pairs")
print(f"(b) P(6,2) = 6! / 4! = 6·5 = {assignments} assignments: 6 choices for the lead, 5 left for the assistant")
print(f"(c) each pair takes the two roles in 2! = {math.factorial(k)} ways: {pairs} × {math.factorial(k)} = {pairs * math.factorial(k)}")
close(pairs, 15); close(assignments, 30); close(assignments / pairs, 2)
print("\nFifteen different pairs can go; thirty different lead–assistant assignments. The 2! underneath in (a) removes exactly those orderings.")

(a) C(6,2) = 6! / (2!·4!) = 6·5 / 2 = 30 / 2 = 15 pairs
(b) P(6,2) = 6! / 4! = 6·5 = 30 assignments: 6 choices for the lead, 5 left for the assistant
(c) each pair takes the two roles in 2! = 2 ways: 15 × 2 = 30

Fifteen different pairs can go; thirty different lead–assistant assignments. The 2! underneath in (a) removes exactly those orderings.


**D2.** Motors from two suppliers: **Rotor** sends 60%, 3% faulty; **Spin** sends 40%, 6% faulty. Last month, 500 motors. (a) The two-way table. (b) $P(\text{faulty} \mid \text{Rotor})$ and $P(\text{Rotor} \mid \text{faulty})$. (c) Independent? (d) Bayes for $P(\text{Spin} \mid \text{faulty})$.

In [21]:
N = 500
counts = [round(.60 * N), round(.40 * N)]                                  # 300 Rotor, 200 Spin
print("(a) " + "; ".join(f"{s}: {c} motors, {round(c*r)} faulty, {c - round(c*r)} good" for s, c, r in zip(["Rotor", "Spin"], counts, [.03, .06])))
d2 = two_way(["Rotor", "Spin"], counts, [.03, .06])
print(d2.to_string())
assert d2.loc["Rotor"].tolist() == [291, 9, 300] and d2.loc["Spin"].tolist() == [188, 12, 200] and d2.loc["total"].tolist() == [479, 21, 500]

p_f_rotor = d2.loc["Rotor", "faulty"] / d2.loc["Rotor", "total"]           # the row
p_rotor_f = d2.loc["Rotor", "faulty"] / d2.loc["total", "faulty"]           # the column
print(f"\n(b) P(faulty | Rotor) = 9 / 300 = {p_f_rotor:.3f}   — Rotor's row: how good is Rotor")
print(f"    P(Rotor | faulty) = 9 / 21  = {p_rotor_f:.3f}   — the faulty column: who sent this faulty one")
close(p_f_rotor, .03); close(p_rotor_f, .429)

p_f_spin = d2.loc["Spin", "faulty"] / d2.loc["Spin", "total"]; p_f = d2.loc["total", "faulty"] / N
print(f"\n(c) rows: Rotor {p_f_rotor:.3f}, Spin {p_f_spin:.3f}; total {p_f:.3f} → the rows differ: DEPENDENT")
close(p_f_spin, .06); close(p_f, .042)

bt2 = bayes_table(["Rotor", "Spin"], [.60, .40], [.03, .06])
print("\n(d)"); print(bt2.round(3).to_string())
p_spin_f = .06 * .40 / (.03 * .60 + .06 * .40)
print(f"    P(Spin | faulty) = .06 × .40 / (.03 × .60 + .06 × .40) = .024 / .042 = {p_spin_f:.3f};  via counts 12/21 = {12/21:.3f}")
close(.60 * .03, .018, 1e-9); close(.40 * .06, .024, 1e-9); close(p_spin_f, .571); close(12 / 21, .571)
print("\nSpin sends 40% of the motors but is behind 57% of the faulty ones: the likelier culprit, not a certain one.")

(a) Rotor: 300 motors, 9 faulty, 291 good; Spin: 200 motors, 12 faulty, 188 good
       good  faulty  total
Rotor   291       9    300
Spin    188      12    200
total   479      21    500

(b) P(faulty | Rotor) = 9 / 300 = 0.030   — Rotor's row: how good is Rotor
    P(Rotor | faulty) = 9 / 21  = 0.429   — the faulty column: who sent this faulty one

(c) rows: Rotor 0.030, Spin 0.060; total 0.042 → the rows differ: DEPENDENT

(d)
       prior share  × defect rate  = faulty, share of all  ÷ total faulty
Rotor          0.6           0.03                   0.018           0.429
Spin           0.4           0.06                   0.024           0.571
total          1.0            NaN                   0.042           1.000
    P(Spin | faulty) = .06 × .40 / (.03 × .60 + .06 × .40) = .024 / .042 = 0.571;  via counts 12/21 = 0.571

Spin sends 40% of the motors but is behind 57% of the faulty ones: the likelier culprit, not a certain one.


**D3.** (a) Warranty claims in the last six weeks: 2, 5, 3, 6, 4, 4 — mean, sample variance and standard deviation by hand, in a table. (b) Saturday brings 10 test rides; three riders in ten buy, each deciding alone: name the model, find $P(2)$, $P(0)$, $P(\geq 1)$, the mean and the standard deviation. (c) Web-shop orders arrive at random, 4 an hour: name the model, find $P(0)$ and $P(2)$ in an hour; then, for a 15-minute coffee break, $\mu$ first, $P(0)$ and $P(\geq 1)$.

In [22]:
xbar3, s2_3 = spread_table([2, 5, 3, 6, 4, 4], label="claims")
close(xbar3, 4); close(s2_3, 2.0); close(math.sqrt(s2_3), 1.41, 5e-3)   # drills.md rounds to two decimals
print(f"\nAbout {xbar3:g} claims a week, give or take {math.sqrt(s2_3):.1f}.")

week  claims  x − x̄  (x − x̄)²
   1     2.0    -2.0        4.0
   2     5.0     1.0        1.0
   3     3.0    -1.0        1.0
   4     6.0     2.0        4.0
   5     4.0     0.0        0.0
   6     4.0     0.0        0.0
        24.0     0.0       10.0

mean   x̄  = 24 / 6 = 4
variance s² = 10 / (6 − 1) = 10 / 5 = 2
st. dev. s  = √2 = 1.41

About 4 claims a week, give or take 1.4.


In [23]:
# D3 (b)
n, p = 10, .3
print("Binomial: a fixed number of trials (10), each yes or no, the same p = .3 every time, each rider alone.\n")
P2 = binomial_steps(n, 2, p)
P0 = (1 - p) ** n
print(f"\nP(0) = .7^10 = {P0:.3f}     P(≥1) = 1 − {P0:.3f} = {1 - P0:.3f}")
print(f"np = 10 × .3 = {n*p:g} buyers     σ = √(10 × .3 × .7) = √{n*p*(1-p):.1f} = {math.sqrt(n*p*(1-p)):.2f}")
close(math.comb(10, 2), 45); close(.7 ** 8, .05765, 1e-4); close(P2, .233)
close(P0, .028); close(1 - P0, .972); close(n * p, 3); close(math.sqrt(n * p * (1 - p)), 1.45)
print("\nExpect three buyers, give or take one and a half; a Saturday with no sale happens about three times in a hundred.")

Binomial: a fixed number of trials (10), each yes or no, the same p = .3 every time, each rider alone.

1. the ways to pick which 2 of the 10 buy:   C(10,2) = 45
2. those 2 buy:                              0.3^2 = 0.090000
3. the other 8 do not:                      0.7^8 = 0.0576
4. multiply:                                 45 × 0.090000 × 0.0576 = 0.233

P(0) = .7^10 = 0.028     P(≥1) = 1 − 0.028 = 0.972
np = 10 × .3 = 3 buyers     σ = √(10 × .3 × .7) = √2.1 = 1.45

Expect three buyers, give or take one and a half; a Saturday with no sale happens about three times in a hundred.


In [24]:
# D3 (c)
print("Poisson: orders arrive at random over time, at a constant rate, with no ceiling on the count.\n")
print("One hour:"); mu4, P0_4 = poisson_steps(4, 1)
print("\nA 15-minute break — a quarter of an hour, so μ first:"); mu1, P0_1 = poisson_steps(4, 0.25)
close(P0_4, .018); close(stats.poisson.pmf(2, 4), .147); close(math.exp(-4), .01832, 1e-4)
close(mu1, 1); close(P0_1, .368); close(1 - P0_1, .632)
print("\nA silent hour is rare, two in a hundred; a silent coffee break happens more than once in three.")

Poisson: orders arrive at random over time, at a constant rate, with no ceiling on the count.

One hour:
μ = 4 an hour × 1 hour(s) = 4
  P(0)  = 4^0 e^−4 / 0! = e^−4 = 0.018
  P(≥1) = 1 − 0.018 = 0.982
  P(2)  = 4² e^−4 / 2! = 16 × 0.0183 / 2 = 0.147

A 15-minute break — a quarter of an hour, so μ first:
μ = 4 an hour × 0.25 hour(s) = 1
  P(0)  = 1^0 e^−1 / 0! = e^−1 = 0.368
  P(≥1) = 1 − 0.368 = 0.632
  P(2)  = 1² e^−1 / 2! = 1 × 0.3679 / 2 = 0.184

A silent hour is rare, two in a hundred; a silent coffee break happens more than once in three.


**D4.** A full charge takes 240 minutes on average, standard deviation 20, bell-shaped: $X \sim N(240, 20)$. (a) Share over 270 minutes. (b) Share between 220 and 260, and the rule of thumb that gives it. (c) The slowest 10% are flagged: from how many minutes? (d) Delivery times: mean 3.1 days, median 2.4 — what shape, and why is the table the wrong tool?

In [25]:
mu, sigma = 240, 20
print("(a) shade the right tail beyond 270"); z_a, above270 = normal_steps(270, mu, sigma, tail="above")
close(z_a, 1.5, 1e-9); close(table_left(z_a), .9332, 1e-4); close(above270, .067)
print("    About 7 charges in 100 take more than four and a half hours.\n")

print("(b) shade between 220 and 260: two lookups, area to the left of each, and a difference")
z_lo, z_hi = (220 - mu) / sigma, (260 - mu) / sigma
between = table_left(z_hi) - table_left(z_lo)
print(f"    z = {z_lo:+.0f} and z = {z_hi:+.0f}:  P(−1 < z < 1) = {table_left(z_hi):.4f} − {table_left(z_lo):.4f} = {between:.3f}")
close(table_left(z_hi), .8413, 1e-4); close(table_left(z_lo), .1587, 1e-4); close(between, .683)
print("    One σ either side of the mean: the 68% rule, no table needed.\n")

print("(c) backwards: the slowest 10% is the right tail, so 90% of the area is to the left of the threshold")
print(f"    find .9000 inside the table: z = 1.28 gives {table_left(1.28):.4f}, z = 1.29 gives {table_left(1.29):.4f} → nearer 1.28   (Excel: =NORM.S.INV(0.90) = {stats.norm.ppf(.9):.3f})")
z_c = 1.28
x_c = mu + z_c * sigma
print(f"    undo the standardisation: x = μ + zσ = 240 + 1.28 × 20 = 240 + {z_c*sigma:.1f} = {x_c:.1f} minutes")
close(table_left(1.28), .8997, 1e-4); close(table_left(1.29), .9015, 1e-4); close(stats.norm.ppf(.9), 1.28, 5e-3); close(x_c, 265.6)
print("    A charge over about 266 minutes is flagged.\n")

print("(d) delivery times: mean 3.1 days, median 2.4 days")
mean_d, median_d = 3.1, 2.4
assert mean_d > median_d
print(f"    mean − median = {mean_d - median_d:+.1f} days: a long right tail, POSITIVE skewness — most parcels in two or three days, a few much later.")
print("    The normal is symmetric (mean = median), so a z from a mean of 3.1 would give the wrong areas: the table only answers questions about bells.")

(a) shade the right tail beyond 270
1. sketch: the bell centred at 240, shade the area above 270
2. standardise: z = (270 − 240) / 20 = 30 / 20 = 1.500 ≈ 1.50
3. table, area to the left of 1.50: 0.9332   (Excel: =NORM.S.DIST(1.50, TRUE))
   the whole area is 1, so P(X > 270) = 1 − 0.9332 = 0.067
4. P(X > 270) = 0.067
    About 7 charges in 100 take more than four and a half hours.

(b) shade between 220 and 260: two lookups, area to the left of each, and a difference
    z = -1 and z = +1:  P(−1 < z < 1) = 0.8413 − 0.1587 = 0.683
    One σ either side of the mean: the 68% rule, no table needed.

(c) backwards: the slowest 10% is the right tail, so 90% of the area is to the left of the threshold
    find .9000 inside the table: z = 1.28 gives 0.8997, z = 1.29 gives 0.9015 → nearer 1.28   (Excel: =NORM.S.INV(0.90) = 1.282)
    undo the standardisation: x = μ + zσ = 240 + 1.28 × 20 = 240 + 25.6 = 265.6 minutes
    A charge over about 266 minutes is flagged.

(d) delivery times: mean 3.1 d

## Homework 7 and what comes next

`homework.md` — due **Sunday 15 November, 23:59 on Moodle**: five exercises on paper and a 150–250-word memo to the Board on the range to print. We go through it together at the start of next Monday's session.

Next Monday, online: *how sure are we of that number?* 200 bikes gave 90 km; the true average is not 90. Week 8 says how far off it can be.

In [26]:
# Every number in drills.md, re-checked here. Change a number in a drill and this cell tells you what else moves.
checks = [
    ("D1 pairs C(6,2)", math.comb(6, 2), 15), ("D1 assignments P(6,2)", math.perm(6, 2), 30),
    ("D2 Rotor faulty", round(.03 * 300), 9), ("D2 Spin faulty", round(.06 * 200), 12), ("D2 total faulty", 9 + 12, 21),
    ("D2 P(faulty | Rotor)", 9 / 300, .03), ("D2 P(Rotor | faulty)", 9 / 21, .429), ("D2 P(faulty | Spin)", 12 / 200, .06),
    ("D2 P(faulty)", 21 / 500, .042), ("D2 Bayes P(Spin | faulty)", .06 * .40 / (.03 * .60 + .06 * .40), .571),
    ("D3 mean", np.mean([2, 5, 3, 6, 4, 4]), 4), ("D3 s²", np.var([2, 5, 3, 6, 4, 4], ddof=1), 2.0), ("D3 s", np.std([2, 5, 3, 6, 4, 4], ddof=1), 1.41),
    ("D3 binomial P(2)", stats.binom.pmf(2, 10, .3), .233), ("D3 binomial P(0)", stats.binom.pmf(0, 10, .3), .028),
    ("D3 binomial P(≥1)", 1 - stats.binom.pmf(0, 10, .3), .972), ("D3 binomial mean", 10 * .3, 3), ("D3 binomial σ", math.sqrt(10 * .3 * .7), 1.45),
    ("D3 Poisson μ=4 P(0)", stats.poisson.pmf(0, 4), .018), ("D3 Poisson μ=4 P(2)", stats.poisson.pmf(2, 4), .147),
    ("D3 Poisson μ=1 P(0)", stats.poisson.pmf(0, 1), .368), ("D3 Poisson μ=1 P(≥1)", 1 - stats.poisson.pmf(0, 1), .632),
    ("D4 P(X > 270)", 1 - stats.norm.cdf(270, 240, 20), .067), ("D4 P(220 < X < 260)", stats.norm.cdf(260, 240, 20) - stats.norm.cdf(220, 240, 20), .683),
    ("D4 flag threshold, minutes", 240 + 1.28 * 20, 265.6),
]
for name, got, want in checks:
    close(got, want, 5e-3 if want < 20 else 0.05)   # three decimals for shares, two for the rest
    print(f"  ok  {name:<32} {got:9.4f}   drills.md: {want}")
print(f"\n{len(checks)} numbers in drills.md, re-checked here: all agree.")

  ok  D1 pairs C(6,2)                    15.0000   drills.md: 15
  ok  D1 assignments P(6,2)              30.0000   drills.md: 30
  ok  D2 Rotor faulty                     9.0000   drills.md: 9
  ok  D2 Spin faulty                     12.0000   drills.md: 12
  ok  D2 total faulty                    21.0000   drills.md: 21
  ok  D2 P(faulty | Rotor)                0.0300   drills.md: 0.03
  ok  D2 P(Rotor | faulty)                0.4286   drills.md: 0.429
  ok  D2 P(faulty | Spin)                 0.0600   drills.md: 0.06
  ok  D2 P(faulty)                        0.0420   drills.md: 0.042
  ok  D2 Bayes P(Spin | faulty)           0.5714   drills.md: 0.571
  ok  D3 mean                             4.0000   drills.md: 4
  ok  D3 s²                               2.0000   drills.md: 2.0
  ok  D3 s                                1.4142   drills.md: 1.41
  ok  D3 binomial P(2)                    0.2335   drills.md: 0.233
  ok  D3 binomial P(0)                    0.0282   drills.md: 0.028
  ok 